In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import arviz as az
import bambi as bmb

from utils import data

In [2]:
df = data.load_individual_features()
df = df.to_pandas()
df.columns

Index(['sequence_id', 'resolution', 'collection_date', 'wn_end_date',
       'wn_start_date', 'wn_prop_sequenced', 'age_band', 'age_group',
       'age_midpoint', 'is_female', 'is_vaccinated', 'pango_lineage',
       'who_voc', 'n_windows', 'wave', 'policy_period', 'policy_period_label',
       'policy_intensity', 'prop_clustered', 'ever_clustered',
       'dz_simd_quintile', 'dz_simd_decile', 'overall_zscore', 'income_zscore',
       'employment_zscore', 'education_zscore', 'health_zscore',
       'access_zscore', 'crime_zscore', 'housing_zscore'],
      dtype='str')

In [3]:
display(df.drop_duplicates(
    subset=['policy_period', 'policy_period_label', 'policy_intensity']
)[['policy_period', 'policy_period_label', 'policy_intensity']])

,policy_period,policy_period_label,policy_intensity
0,OM,Omicron wave,42
1,FE,Final easing,15
3,F5,Five-tier framework,65
4,NN,Near-normal,10
6,L21,Level 2 / Level 1,38
12,PR,Post-restriction,3
21,L2,Second lockdown,95
25,SL,Stay local — Level 3,65
30,T1,Pre-tier tightening,55
54,P3,Route map phase 3,30


In [10]:
pr = df[
    (df["resolution"] == data.PRIMARY_RESOLUTION)
    & (df["policy_period"] == "PR")
].copy()

pr["wn_prop_sequenced100"] = pr["wn_prop_sequenced"] * 100

# # Set reference levels by making them the first category
# existing = pr["dz_simd_quintile"].astype("category").cat.categories.tolist()
# simd_ref = 3
# simd = [simd_ref] + [c for c in existing if c != simd_ref]
# pr["dz_simd_quintile"] = pr["dz_simd_quintile"].astype("category").cat.reorder_categories(simd)
#
# age_groups = pr["age_group"].astype("category").cat.categories.tolist()
# age_ref = "40-59"
# ages = [age_ref] + [c for c in age_groups if c != age_ref]
# pr["age_group"] = pr["age_group"].astype("category").cat.reorder_categories(ages)
#
# pr["ever_clustered"] = pr["ever_clustered"].astype(int)
# pr["is_female"]      = pr["is_female"].astype(int)

In [13]:
f1 = """
    ever_clustered ~
    C(dz_simd_quintile, Treatment(3)) +
    C(age_group, Treatment('40-59')) +
    is_female +
    logit_prop_seq
"""

m1 = smf.logit(f1, data=pr).fit()
print(m1.summary())

or_table = np.exp(m1.conf_int())
or_table["OR"] = np.exp(m1.params)
or_table.columns = ["2.5%", "97.5%", "OR"]
print(or_table[["OR", "2.5%", "97.5%"]])

Optimization terminated successfully.
         Current function value: 0.452427
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:         ever_clustered   No. Observations:                41737
Model:                          Logit   Df Residuals:                    41725
Method:                           MLE   Df Model:                           11
Date:                Thu, 30 Apr 2026   Pseudo R-squ.:                 0.01403
Time:                        16:31:21   Log-Likelihood:                -18883.
converged:                       True   LL-Null:                       -19152.
Covariance Type:            nonrobust   LLR p-value:                3.507e-108
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                                 

In [ ]:
formula = """
ever_clustered ~ C(age_group, Treatment('40-59'))
    + is_female
    + C(dz_simd_quintile, Treatment(3))
    + is_vaccinated
    + seq_prop_zscore
"""


for wave in waves:
    print(f"Wave: {wave} ================================================================")

    model_data = df[
        (df["resolution"] == data.PRIMARY_RESOLUTION)
        & (df["wave"] == wave)
    ]

    fit = smf.logit(
        formula = formula,
        data    = model_data,
    ).fit()

    print(fit.summary())

In [ ]:
model_data = df[df["resolution"] == data.PRIMARY_RESOLUTION]

age_order = ['40-59', '00-09', '10-19', '20-39', '60-74', 'elderly']  # reference first
model_data['age_group'] = pd.Categorical(
    model_data['age_group'],
    categories=age_order,
    ordered=False
)

model_data['dz_simd_quintile'] = pd.Categorical(
    model_data['dz_simd_quintile'],
    categories=[3, 1, 2, 4, 5],  # reference = 3
    ordered=False
)

formula = """
ever_clustered ~ age_group
    + is_female
    + dz_simd_quintile
    + is_vaccinated
    + seq_prop_zscore
"""

model = bmb.Model(
    formula=formula,
    data=model_data,
    family="bernoulli",
)

idata = model.fit(
    random_seed=42,
    inference_method="nutpie",
    nuts_kwargs={"target_accept": 0.95},
    tune=2000,
)

In [ ]:
az.plot_forest(idata, combined=True)
plt.axvline(0, color="black", ls="--", lw=0.5)